# Main DLSTM: Final Model
## Data preparation and server check
### This notebook will not run any huge process. First you need to check dependencies based on <b>READEME</b> file. 

#### This model is redesigned based on research considerations and isn't an exact copy of the base article codes. In the "docs/Method_Decisions.md" you will find the key areas which are differ from base article.

In [ ]:
from pathlib import Path
import sys
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "thesis").is_dir()), None)
if PROJECT is None:
    raise FileNotFoundError("Extract the complete project folder; do not copy the notebook alone.")
if str(PROJECT) not in sys.path: 
    sys.path.insert(0, str(PROJECT))
from thesis.common import config
CONFIG = "configs/default.json"  # all notebooks must use the SAME config
c = config(CONFIG)
print("Experiment:", c["experiment_name"], "| outputs:", c["output_root"])


#### Checking environment attributes and network size:  

In [ ]:
from thesis.pipeline import preflight
preflight(c)

### Optional Windows server benchmark
Set RUN_BENCHMARK=True to run the full-width, training-only timing diagnostic.
It uses configs/default.json and does not produce final test results.

In [ ]:
RUN_BENCHMARK = False
if RUN_BENCHMARK:
    from thesis.benchmark import benchmark
    benchmark_path = benchmark(config("configs/default.json"))
    print(benchmark_path / "Benchmark_Report.md")


#### Creating completed sequences (Company-Years) which have atleast 100 days of trading data: 

In [ ]:
from thesis.data import prepare
import json, pandas as pd
prepared=prepare(c)
metadata = json.loads(
    (prepared / "metadata.json").read_text(encoding="utf-8")
)

print(json.dumps(metadata, ensure_ascii=False, indent=2))
samples=pd.read_csv(prepared/"samples.csv")
display(samples.groupby("year").agg(
    samples=("sample_id","size"),
    mean_days=("length","mean"),
    min_days=("length","min"),
    max_days=("length","max")
))

#### Recording necessary results on the notebook output:

In [ ]:
from thesis.recovery import emit_capsule
emit_capsule([prepared/"metadata.json",prepared/"samples.csv",prepared/"excluded.csv"])

#### Independent Backup cell
##### This cell will save the results based on notebook output cells. Wieghts and biases still need "model.pt" to resolve the previous results as backup. 
##### Also when <code>RESTORE = False</code> backup process will not be run and if you want to save a backup from outputs, you need to change it to <i><u>True</u></i>.

In [ ]:
RESTORE = False
NOTEBOOK_DIR = Path.cwd() if "Path" in globals() else __import__("pathlib").Path.cwd()
RECOVERY_DESTINATION = NOTEBOOK_DIR / "recovered_outputs"

def restore_from_notebooks(notebook_dir, destination):
    from pathlib import Path
    import base64,hashlib,json,re,zlib
    destination=Path(destination).resolve()
    destination.mkdir(parents=True,exist_ok=True)
    restored=[]
    for nb in sorted(Path(notebook_dir).rglob('*.ipynb')):
        notebook=json.loads(nb.read_text(encoding='utf-8'))
        texts=[]
        for cell in notebook.get('cells',[]):
            pieces=[]
            for output in cell.get('outputs',[]):
                text=output.get('text',output.get('data',{}).get('text/plain',''))
                pieces.append(''.join(text) if isinstance(text,list) else text)
            texts.append(''.join(pieces))
        for text in texts:
            for match in re.finditer(r'THESIS_CAPSULE_BEGIN ([a-f0-9]{64})\s+([A-Za-z0-9+/=\s]+?)THESIS_CAPSULE_END',text):
                raw=zlib.decompress(base64.b64decode(''.join(match[2].split())))
                if hashlib.sha256(raw).hexdigest()!=match[1]:
                    raise ValueError('Capsule checksum mismatch')
                
                archive=json.loads(raw)
                for name,item in archive['files'].items():
                    p=(destination/name).resolve()
                    if not p.is_relative_to(destination):
                        raise ValueError('Unsafe capsule path')
                    
                    b=base64.b64decode(item['data'])
                    if hashlib.sha256(b).hexdigest()!=item['sha256']:
                        raise ValueError('File checksum mismatch')
                    
                    if p.exists() and p.read_bytes()!=b:
                        # Preserve both historical snapshots; do not silently overwrite conflicting evidence.
                        p=p.with_name(p.name+'.snapshot_'+item['sha256'][:12])
                    p.parent.mkdir(parents=True,exist_ok=True)
                    p.write_bytes(b)
                    restored.append(str(p))
    print(f'Restored {len(set(restored))} files. Weights require model.pt/checkpoint backups.')
    return sorted(set(restored))

if RESTORE:
    restored = restore_from_notebooks(NOTEBOOK_DIR, RECOVERY_DESTINATION)
